In [8]:
from typing import List, Optional

class Node:
    def __init__(self, is_leaf: bool):
        self.is_leaf = is_leaf
        self.parent: Optional['InternalNode'] = None

class LeafNode(Node):
    def __init__(self):
        super().__init__(is_leaf=True)
        self.keys: List[int] = []
        self.next: Optional['LeafNode'] = None

    def __repr__(self):
        return f"Leaf({self.keys})"

class InternalNode(Node):
    def __init__(self):
        super().__init__(is_leaf=False)
        self.keys: List[int] = []
        self.children: List[Node] = []  #leng = len(keys)+1

    def __repr__(self):
        return f"Internal({self.keys})"


class BPlusTree:
    def __init__(self, d: int):
        self.d = d
        self.root: Node = LeafNode() #start with empty leaf as root

    

    def _find_leaf(self, key: int) -> LeafNode:
        node = self.root
        while not node.is_leaf:
            inode: InternalNode = node  
            # choose child by keys
            i = 0
            while i < len(inode.keys) and key >= inode.keys[i]:
                i += 1
            node = inode.children[i]
        return node  



    def _insert_into_parent(self, left: Node, key_for_parent: int, right: Node):
        # Insert key_for_parent into parent of left; if parent overflows split recursively
        if left.parent is None:
            # create new root
            new_root = InternalNode()
            new_root.keys = [key_for_parent]
            new_root.children = [left, right]
            left.parent = new_root
            right.parent = new_root
            self.root = new_root
            return

        parent: InternalNode = left.parent  # type: ignore
        # find left index in parent.children
        idx = parent.children.index(left)
        parent.children.insert(idx+1, right)
        parent.keys.insert(idx, key_for_parent)
        right.parent = parent

        # overflow?
        if len(parent.keys) > 2 * self.d:
            self._split_internal(parent)

    def _split_leaf(self, leaf: LeafNode):
        # leaf has len = 2d+1 after insertion
        total = len(leaf.keys)
        assert total == 2 * self.d + 1
        left_count = self.d
        right_keys = leaf.keys[left_count:]
        left_keys = leaf.keys[:left_count]

        new_leaf = LeafNode()
        new_leaf.keys = right_keys
        leaf.keys = left_keys

        # link
        new_leaf.next = leaf.next
        leaf.next = new_leaf

        # parent handling: copy-up the first key of new_leaf
        key_to_parent = new_leaf.keys[0]
        parent = leaf.parent
        if parent is None:
            # new root
            new_root = InternalNode()
            new_root.keys = [key_to_parent]
            new_root.children = [leaf, new_leaf]
            leaf.parent = new_root
            new_leaf.parent = new_root
            self.root = new_root
        else:
            # insert key_to_parent to parent (copy up)
            self._insert_into_parent(leaf, key_to_parent, new_leaf)

    def _split_internal(self, inode: InternalNode):
        # inode has len(keys) = 2d+1 after insertion
        total = len(inode.keys)
        assert total == 2 * self.d + 1
        mid_index = self.d  # push up the middle key at index d (0-based)
        push_up_key = inode.keys[mid_index]

        # left and right parts
        left_keys = inode.keys[:mid_index]
        right_keys = inode.keys[mid_index+1:]

        left_children = inode.children[:mid_index+1]
        right_children = inode.children[mid_index+1:]

        left_node = InternalNode()
        left_node.keys = left_keys
        left_node.children = left_children
        for c in left_children:
            c.parent = left_node

        right_node = InternalNode()
        right_node.keys = right_keys
        right_node.children = right_children
        for c in right_children:
            c.parent = right_node

        parent = inode.parent
        if parent is None:
            # create new root
            new_root = InternalNode()
            new_root.keys = [push_up_key]
            new_root.children = [left_node, right_node]
            left_node.parent = new_root
            right_node.parent = new_root
            self.root = new_root
        else:
            # replace inode in parent with left_node and right_node and insert push_up_key
            idx = parent.children.index(inode)
            parent.children[idx] = left_node
            parent.children.insert(idx+1, right_node)
            parent.keys.insert(idx, push_up_key)
            left_node.parent = parent
            right_node.parent = parent
            # if parent overflows, split recursively
            if len(parent.keys) > 2 * self.d:
                self._split_internal(parent)

    # ---------- public operations ----------
    def insert(self, key: int):
        leaf = self._find_leaf(key)
        # simple insert sorted (no duplicate handling specifics; we ignore duplicates)
        import bisect
        pos = bisect.bisect_left(leaf.keys, key)
        if pos < len(leaf.keys) and leaf.keys[pos] == key:
            # already present: ignore or you could allow duplicates; we'll ignore
            return
        leaf.keys.insert(pos, key)
        # overflow?
        if len(leaf.keys) > 2 * self.d:
            self._split_leaf(leaf)

    def _borrow_from_left(self, leaf: LeafNode, left_sib: LeafNode, parent: InternalNode, idx_in_parent: int) -> bool:
        # left_sib must exist
        if len(left_sib.keys) > self.d:
            # borrow last key of left sibling
            borrowed = left_sib.keys.pop(-1)
            leaf.keys.insert(0, borrowed)
            # update parent's separator key for these two children
            # parent.keys[idx_in_parent] is separator (between left_sib and leaf)
            parent.keys[idx_in_parent] = leaf.keys[0]  # copy up first key of leaf
            return True
        return False

    def _borrow_from_right(self, leaf: LeafNode, right_sib: LeafNode, parent: InternalNode, idx_in_parent: int) -> bool:
        # right_sib must exist
        if len(right_sib.keys) > self.d:
            borrowed = right_sib.keys.pop(0)
            leaf.keys.append(borrowed)
            # parent key between leaf and right_sib should be updated to right_sib.first
            parent.keys[idx_in_parent] = right_sib.keys[0]
            return True
        return False

    def _merge_leaves(self, left: LeafNode, right: LeafNode, parent: InternalNode, idx_in_parent: int):
        # merge right into left (left.keys extend)
        left.keys.extend(right.keys)
        left.next = right.next
        # remove pointer and separator key from parent
        parent.children.pop(idx_in_parent+1)  # remove right
        parent.keys.pop(idx_in_parent)        # remove separator
        # if parent becomes empty and is root, adjust root
        self._fix_internal_after_delete(parent)

    def _fix_internal_after_delete(self, inode: InternalNode):
        # if inode is root and empty
        if inode.parent is None:
            if len(inode.keys) == 0:
                # shrink tree height: make single child the root
                child = inode.children[0]
                child.parent = None
                self.root = child
            return

        if len(inode.keys) < self.d:
            # try to borrow from siblings (internal node borrow)
            parent = inode.parent
            idx = parent.children.index(inode)
            # try left sibling
            left_sib = None
            right_sib = None
            if idx - 1 >= 0:
                left_sib = parent.children[idx-1]
            if idx + 1 < len(parent.children):
                right_sib = parent.children[idx+1]

            # Borrow from left internal sibling
            if left_sib and not left_sib.is_leaf and len(left_sib.keys) > self.d:
                ls: InternalNode = left_sib  # type: ignore
                # rotate via parent: bring parent's separator down
                borrow_key = ls.keys.pop(-1)
                borrow_child = ls.children.pop(-1)
                inode.keys.insert(0, parent.keys[idx-1])
                parent.keys[idx-1] = borrow_key
                inode.children.insert(0, borrow_child)
                borrow_child.parent = inode
                return
            # Borrow from right sibling
            if right_sib and not right_sib.is_leaf and len(right_sib.keys) > self.d:
                rs: InternalNode = right_sib  # type: ignore
                borrow_key = rs.keys.pop(0)
                borrow_child = rs.children.pop(0)
                inode.keys.append(parent.keys[idx])
                parent.keys[idx] = borrow_key
                inode.children.append(borrow_child)
                borrow_child.parent = inode
                return
            # Otherwise merge
            if left_sib and not left_sib.is_leaf:
                # merge inode into left_sib
                ls: InternalNode = left_sib  # type: ignore
                # pull down parent's separator
                sep = parent.keys.pop(idx-1)
                parent.children.pop(idx)  # remove inode
                ls.keys.append(sep)
                ls.keys.extend(inode.keys)
                ls.children.extend(inode.children)
                for c in inode.children:
                    c.parent = ls
                # recursive fix
                self._fix_internal_after_delete(parent)
                return
            if right_sib and not right_sib.is_leaf:
                # merge right_sib into inode
                rs: InternalNode = right_sib  # type: ignore
                sep = parent.keys.pop(idx)
                parent.children.pop(idx+1)
                inode.keys.append(sep)
                inode.keys.extend(rs.keys)
                inode.children.extend(rs.children)
                for c in rs.children:
                    c.parent = inode
                self._fix_internal_after_delete(parent)
                return

    def delete(self, key: int):
        leaf = self._find_leaf(key)
        if key not in leaf.keys:
            return  # not present, ignore
        leaf.keys.remove(key)

        # if leaf is root (only node) -> done
        if leaf.parent is None:
            # if empty, keep empty root leaf
            return

        # if still enough keys
        if len(leaf.keys) >= self.d:
            # if first key changed, need to update parent's separator (the parent's entry that points to this leaf)
            parent = leaf.parent
            # find index of leaf in parent children
            idx = parent.children.index(leaf)
            # If leaf is not the leftmost child, parent's key at idx-1 should be <= leaf.first. But simpler:
            # For B+ trees we keep parent separators as the first key of the right child.
            # So if idx>0, parent.keys[idx-1] should equal leaf.keys[0] (if leaf is right child). Update if necessary.
            if idx > 0:
                parent.keys[idx-1] = leaf.keys[0]
            return

        # leaf underflows (len < d). Try to borrow from siblings (same parent)
        parent = leaf.parent
        idx = parent.children.index(leaf)
        left_sib = parent.children[idx-1] if idx-1 >= 0 else None
        right_sib = parent.children[idx+1] if idx+1 < len(parent.children) else None

        # Try borrow from left sibling
        if left_sib and left_sib.is_leaf:
            if self._borrow_from_left(leaf, left_sib, parent, idx-1):
                return
        # Try borrow from right sibling
        if right_sib and right_sib.is_leaf:
            if self._borrow_from_right(leaf, right_sib, parent, idx):
                return

        # Otherwise merge
        # Prefer merge with left sibling if exists, else with right sibling
        if left_sib and left_sib.is_leaf:
            self._merge_leaves(left_sib, leaf, parent, idx-1)
            return
        if right_sib and right_sib.is_leaf:
            self._merge_leaves(leaf, right_sib, parent, idx)
            return

    # ---------- init and show ----------
    def init_tree_from_structure(self, root_node: Node):
        # Initialize tree directly (useful for demo initial state)
        self.root = root_node
        # set parents correctly via DFS
        def dfs_set_parents(node, parent):
            node.parent = parent
            if node.is_leaf:
                return
            inode: InternalNode = node  # type: ignore
            for c in inode.children:
                dfs_set_parents(c, inode)
        dfs_set_parents(self.root, None)

    def show(self):
        # Print tree by levels (BFS)
        from collections import deque
        q = deque()
        q.append(self.root)
        level = 0
        out_lines = []
        while q:
            sz = len(q)
            level_nodes = []
            for _ in range(sz):
                n = q.popleft()
                if n.is_leaf:
                    level_nodes.append(f"L{n.keys}")
                else:
                    level_nodes.append(f"I{n.keys}")
                    inode: InternalNode = n  # type: ignore
                    for c in inode.children:
                        q.append(c)
            out_lines.append(f"Level {level}: " + " | ".join(level_nodes))
            level += 1


        print("\n".join(out_lines))
        print("-" * 60)


def d1_sequence():
    
    tree = BPlusTree(d=1)

    # Build leaves as per the image initial state:
    # Leaves: L1 = [13], L2 = [16,19], L3 = [20,25]
    L1 = LeafNode(); L1.keys = [13]
    L2 = LeafNode(); L2.keys = [16,19]
    L3 = LeafNode(); L3.keys = [20,25]
    L1.next = L2; L2.next = L3

    # Build root internal node with keys [16,20] and children L1,L2,L3
    root = InternalNode()
    root.keys = [16,20]
    root.children = [L1, L2, L3]
    L1.parent = root; L2.parent = root; L3.parent = root

    tree.init_tree_from_structure(root)

    print("Initial tree:")
    tree.show()

    ops = [
        ("insert", 27),  # Q1
        ("insert", 14),  # Q2
        ("insert", 22),  # Q3 part1
        ("insert", 21),  # Q3 part2
        ("delete", 21),  # Q4
        ("delete", 22),  # Q5
        ("delete", 20),  # Q6
        ("delete", 25),  
    ]

    for op, k in ops:
        print(f"{op.upper()} {k} ...")
        if op == "insert":
            tree.insert(k)
        else:
            tree.delete(k)
        tree.show()

if __name__ == "__main__":
    d1_sequence()


Initial tree:
Level 0: I[16, 20]
Level 1: L[13] | L[16, 19] | L[20, 25]
------------------------------------------------------------
INSERT 27 ...
Level 0: I[20]
Level 1: I[16] | I[25]
Level 2: L[13] | L[16, 19] | L[20] | L[25, 27]
------------------------------------------------------------
INSERT 14 ...
Level 0: I[20]
Level 1: I[16] | I[25]
Level 2: L[13, 14] | L[16, 19] | L[20] | L[25, 27]
------------------------------------------------------------
INSERT 22 ...
Level 0: I[20]
Level 1: I[16] | I[25]
Level 2: L[13, 14] | L[16, 19] | L[20, 22] | L[25, 27]
------------------------------------------------------------
INSERT 21 ...
Level 0: I[20]
Level 1: I[16] | I[21, 25]
Level 2: L[13, 14] | L[16, 19] | L[20] | L[21, 22] | L[25, 27]
------------------------------------------------------------
DELETE 21 ...
Level 0: I[20]
Level 1: I[16] | I[22, 25]
Level 2: L[13, 14] | L[16, 19] | L[20] | L[22] | L[25, 27]
------------------------------------------------------------
DELETE 22 ...
Level

In [9]:
def d2_sequence():
    # d = 2 (minimum degree)
    tree = BPlusTree(d=2)

    # Build leaves as initial state:
    # Leaves: L1 = [5, 10], L2 = [20, 30], L3 = [40, 50]
    L1 = LeafNode(); L1.keys = [5, 10]
    L2 = LeafNode(); L2.keys = [20, 30]
    L3 = LeafNode(); L3.keys = [40, 50]
    L1.next = L2; L2.next = L3

    # Build root internal node with keys [20, 40] and children L1, L2, L3
    root = InternalNode()
    root.keys = [20, 40]
    root.children = [L1, L2, L3]
    L1.parent = root; L2.parent = root; L3.parent = root

    tree.init_tree_from_structure(root)

    print("Initial tree:")
    tree.show()

    ops = [
        ("insert", 25),  # goes into L2 -> no split (becomes 20,25,30)
        ("insert", 15),  # goes into L1 -> no split (becomes 5,10,15)
        ("insert", 35),  # goes into L2 -> becomes 20,25,30,35 (max = 4)
        ("insert", 45),  # goes into L3 -> becomes 40,45,50
        ("insert", 27),  # causes L2 overflow -> split into [20,25] and [27,30,35], parent gains key 27
        ("delete", 10),  # remove from left leaf -> stays >= min
        ("delete", 5),   # underflows left leaf -> merge with sibling [20,25] -> [15,20,25], parent key 20 removed
        ("delete", 27),  # remove 27 from middle leaf -> update parent separator to new first key (30)
        ("delete", 30),  # causes underflow in middle -> borrow from left (transfer 25), parent updated to 25
        ("delete", 45),
        ("delete", 35),
    ]

    for op, k in ops:
        print(f"{op.upper()} {k} ...")
        if op == "insert":
            tree.insert(k)
        else:
            tree.delete(k)
        tree.show()

if __name__ == "__main__":
    d2_sequence()


Initial tree:
Level 0: I[20, 40]
Level 1: L[5, 10] | L[20, 30] | L[40, 50]
------------------------------------------------------------
INSERT 25 ...
Level 0: I[20, 40]
Level 1: L[5, 10] | L[20, 25, 30] | L[40, 50]
------------------------------------------------------------
INSERT 15 ...
Level 0: I[20, 40]
Level 1: L[5, 10, 15] | L[20, 25, 30] | L[40, 50]
------------------------------------------------------------
INSERT 35 ...
Level 0: I[20, 40]
Level 1: L[5, 10, 15] | L[20, 25, 30, 35] | L[40, 50]
------------------------------------------------------------
INSERT 45 ...
Level 0: I[20, 40]
Level 1: L[5, 10, 15] | L[20, 25, 30, 35] | L[40, 45, 50]
------------------------------------------------------------
INSERT 27 ...
Level 0: I[20, 27, 40]
Level 1: L[5, 10, 15] | L[20, 25] | L[27, 30, 35] | L[40, 45, 50]
------------------------------------------------------------
DELETE 10 ...
Level 0: I[20, 27, 40]
Level 1: L[5, 15] | L[20, 25] | L[27, 30, 35] | L[40, 45, 50]
-----------------